In [11]:
!nvidia-smi

Thu Jun 18 21:59:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
%cd /content/CUDAFUNCTIONS
!rm -rf build
!mkdir build
%cd build
!cmake ..
!make

/content/CUDAFUNCTIONS
/content/CUDAFUNCTIONS/build
-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Configuring done (3.8s)
CMake Warning (dev) in CMakeLists.txt:
  Policy CMP0104 is not set: CMAKE_CUDA_ARCHITECTURES now detected for NVCC,
  empty CUDA_ARCHITECTURES not allowed.  Run "cmake --help-policy CMP0104"
  for policy details.  Use the cmake_policy command to set the policy and
  suppress this warning.

  CUDA_ARCHITECTURES is empty for target "vector_add".
This 

In [13]:
%%writefile /content/CUDAFUNCTIONS/python/ctypes/test_ctypes.py
import ctypes
import numpy as np

lib = ctypes.CDLL("/content/CUDAFUNCTIONS/build/libvector_add.so")

lib.vectorAdd.argtypes = [
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.c_int
]

lib.vectorAdd.restype = None

N = 10

a = np.arange(N, dtype=np.int32)
b = np.arange(N, 2 * N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

lib.vectorAdd(
    a.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    b.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    c.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    N
)

print("Method 1: ctypes")
print("Input A:", a)
print("Input B:", b)
print("Result :", c)

Writing /content/CUDAFUNCTIONS/python/ctypes/test_ctypes.py


In [14]:
%cd /content/CUDAFUNCTIONS/python/ctypes
!python3 test_ctypes.py

/content/CUDAFUNCTIONS/python/ctypes
Method 1: ctypes
Input A: [0 1 2 3 4 5 6 7 8 9]
Input B: [10 11 12 13 14 15 16 17 18 19]
Result : [10 12 14 16 18 20 22 24 26 28]


In [15]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper/vector_add_wrapper.c
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <dlfcn.h>
#include <stdlib.h>

typedef void (*vectorAddFunc)(int *, int *, int *, int);

static vectorAddFunc vectorAdd = NULL;

static PyObject* pyVectorAdd(PyObject* self, PyObject* args) {
    PyObject *py_a, *py_b;
    int N;

    if (!PyArg_ParseTuple(args, "OOi", &py_a, &py_b, &N))
        return NULL;

    if (!PyList_Check(py_a) || !PyList_Check(py_b) ||
        PyList_Size(py_a) != N || PyList_Size(py_b) != N) {
        PyErr_SetString(PyExc_ValueError, "Arguments must be lists of size N.");
        return NULL;
    }

    int *a = malloc(N * sizeof(int));
    int *b = malloc(N * sizeof(int));
    int *c = malloc(N * sizeof(int));

    for (int i = 0; i < N; i++) {
        a[i] = (int)PyLong_AsLong(PyList_GetItem(py_a, i));
        b[i] = (int)PyLong_AsLong(PyList_GetItem(py_b, i));
    }

    vectorAdd(a, b, c, N);

    PyObject *result = PyList_New(N);

    for (int i = 0; i < N; i++) {
        PyList_SetItem(result, i, PyLong_FromLong(c[i]));
    }

    free(a);
    free(b);
    free(c);

    return result;
}

static PyMethodDef VectorAddMethods[] = {
    {"vectorAdd", pyVectorAdd, METH_VARARGS, "Run CUDA vector addition"},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef vectoraddmodule = {
    PyModuleDef_HEAD_INIT,
    "vector_add_wrapper",
    NULL,
    -1,
    VectorAddMethods
};

PyMODINIT_FUNC PyInit_vector_add_wrapper(void) {
    void *handle = dlopen(
        "/content/CUDAFUNCTIONS/build/libvector_add.so",
        RTLD_LAZY
    );

    if (!handle) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    vectorAdd = (vectorAddFunc)dlsym(handle, "vectorAdd");

    if (!vectorAdd) {
        PyErr_SetString(PyExc_ImportError, dlerror());
        return NULL;
    }

    return PyModule_Create(&vectoraddmodule);
}

Writing /content/CUDAFUNCTIONS/python/wrapper/vector_add_wrapper.c


In [16]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper/setup.py
from setuptools import setup, Extension

module = Extension(
    "vector_add_wrapper",
    sources=["vector_add_wrapper.c"],
    libraries=["dl"]
)

setup(
    name="vector_add_wrapper",
    version="1.0",
    ext_modules=[module]
)

Writing /content/CUDAFUNCTIONS/python/wrapper/setup.py


In [17]:
%%writefile /content/CUDAFUNCTIONS/python/wrapper/test_vector_add_wrapper.py
import vector_add_wrapper as vaw

N = 10

a = list(range(N))
b = list(range(N, 2 * N))

result = vaw.vectorAdd(a, b, N)

print("Method 2: Python C wrapper")
print("Input A:", a)
print("Input B:", b)
print("Result :", result)

Writing /content/CUDAFUNCTIONS/python/wrapper/test_vector_add_wrapper.py


In [18]:
%cd /content/CUDAFUNCTIONS/python/wrapper
!python3 setup.py build_ext --inplace
!python3 test_vector_add_wrapper.py

/content/CUDAFUNCTIONS/python/wrapper
running build_ext
building 'vector_add_wrapper' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/include/python3.12 -c vector_add_wrapper.c -o build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-gcc -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/vector_add_wrapper.o -L/usr/lib/x86_64-linux-gnu -ldl -o build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/vector_add_wrapper.cpython-312-x86_64-linux-gnu.so -> 
Method 2: Python C wrapper
Input A: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Input B: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Result : [10, 12, 14, 16, 18, 20, 22, 24, 26, 28